# Reconhecimento de Pessoas com CNN

## Objetivo

- Coletar uma base própria de rostos dos integrantes do grupo (uma pasta por pessoa) com o script `src/captura_faces.py` e treinar uma **CNN** para reconhecer cada um.
- Adicionar a classe **"Desconhecido"** com rostos de pessoas que **não** são do grupo: base **LFW** via `scikit-learn` e as pastas de `dados/desconhecidos/` (fotos de galeria e gente de fora capturada com a mesma webcam).
- Dividir os dados em **70% treino / 30% teste**, de forma estratificada e sem vazamento, e avaliar com **matriz de confusão** e métricas de **acurácia, precisão, recall e F1**.
- Salvar o modelo treinado (`modelos/modelo_rostos.keras`), usado pela UI web (`src/app_demo.py`) e pela demo OpenCV (`src/demo_reconhecimento.py`).

### Estrutura do projeto

```text
atividade_cnn_rostos.ipynb   este notebook: dados, treino, avaliação
src/
    app_demo.py              UI web: câmera ao vivo + upload de foto
    demo_reconhecimento.py   demo OpenCV
    captura_faces.py         coleta pela webcam
    extrair_video.py         coleta a partir de vídeo
    recortar_galeria.py      coleta a partir de fotos
    deteccao.py              YuNet, alinhamento e pré-processamento
    emocao.py                modelo de expressão pré-treinado
    config.py                caminhos e parâmetros
    verificar_modelo.py      confere modelo + classes
    templates/demo.html
    static/demo.css
    static/demo.js
dados/
    integrantes/<nome>/      uma pasta por integrante = uma classe
    desconhecidos/<origem>/  gente de fora = classe Desconhecido
modelos/                     modelo treinado e modelos baixados (ignorado no Git)
```

### Como rodar (da raiz do projeto)

1. `python src/captura_faces.py <nome>` para cada integrante. Use `--desconhecido <nome>` para pessoas de fora. Opcionalmente, `python src/extrair_video.py <video> <nome>` ou `python src/recortar_galeria.py <pasta> <nome> --integrante`.
2. Execute este notebook de cima para baixo. O LFW é baixado na primeira execução.
3. `python src/verificar_modelo.py` confere o modelo e o `classes.txt`.
4. `python src/app_demo.py` abre a UI web em `http://127.0.0.1:5000`.
5. `python src/demo_reconhecimento.py --emocao` abre a demo OpenCV.

### Em outro computador

Clone o repositório, crie um ambiente com Python 3.11 e rode `pip install -r requirements.txt`. As fotos e o modelo treinado não vêm no Git: colete as fotos e rode o notebook. Se o grupo enviar `modelo_treinado.zip`, copie `modelo_rostos.keras` e `classes.txt` para `modelos/` antes de rodar a demo.


In [ ]:
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.datasets import fetch_lfw_people, get_data_home
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# caminhos, parâmetros, detecção, alinhamento e pré-processamento: os MESMOS da
# captura e da demo (src/config.py e src/deteccao.py)
sys.path.insert(0, str(Path("src").resolve()))
from config import (
    ARQUIVO_CLASSES, ARQUIVO_MODELO, IMG_SIZE, NOME_MODELO, PASTA_DESCONHECIDOS, PASTA_INTEGRANTES,
)
from deteccao import alinhar, criar_detector, detectar_rostos, ler_imagem, preparar_para_cnn

plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

# ----------------------------------------------------------------------
# Parâmetros do experimento (tudo em um lugar só = fácil reproduzir)
# ----------------------------------------------------------------------
SEED = 42
# IMG_SIZE (64x64) vem de src/config.py: a demo precisa usar o mesmo valor
TEST_SIZE = 0.30           # 70% treino / 30% teste
VAL_SIZE = 0.20            # fração do treino usada como validação
N_DESCONHECIDO_LFW = 1500  # rostos do LFW usados na classe "Desconhecido"
LFW_POR_PESSOA = 2         # no máximo 2 fotos por pessoa: mais identidades diferentes
TAMANHO_BLOCO = 20         # fotos seguidas da webcam (~7 s) ficam juntas na divisão
BATCH_SIZE = 32
EPOCAS = 60

PASTA_LFW = PASTA_DESCONHECIDOS / "lfw"
if not PASTA_INTEGRANTES.exists():
    raise FileNotFoundError(
        "Pasta 'dados/integrantes/' não encontrada. Colete as fotos primeiro com: "
        "python src/captura_faces.py <nome>"
    )

# cada pasta de dados/integrantes/ é uma classe; tudo em dados/desconhecidos/ (LFW,
# WIDER, galeria, webcam) vai para a classe Desconhecido
NOMES = sorted(pasta.name for pasta in PASTA_INTEGRANTES.iterdir() if pasta.is_dir())
CLASSES = NOMES + ["Desconhecido"]

# semente de treino: sem isto, duas execucoes do mesmo notebook geram modelos
# diferentes -- e a classe que absorve rostos de fora muda de uma para outra.
keras.utils.set_random_seed(SEED)

print("Integrantes encontrados:", NOMES)

## 1. Base de dados

- **Integrantes**: fotos capturadas pela webcam ou extraídas de vídeo (`dados/integrantes/<nome>/`).
- **Desconhecido**: rostos de pessoas que **não** são do grupo — 1500 do **LFW** (até 2 por pessoa, ~750 pessoas reais, sem download manual) mais as outras pastas de `dados/desconhecidos/`.

**Toda foto, de qualquer origem, passa pela mesma cadeia** (`src/deteccao.py`, usada também pela captura e pela demo):

1. **Detecção com YuNet** (rede neural do OpenCV): acha o rosto mesmo virado para o lado ou para cima/baixo, onde o Haar cascade frontal da primeira versão já falhava; devolve 5 pontos (olhos, nariz e cantos da boca).
2. **Alinhamento**: gira e escala o rosto para os 5 pontos caírem sempre na mesma posição do recorte (112x112). Tira inclinação e distância da conta e corta o fundo.
3. **Tons de cinza 64x64 + CLAHE** (equalização local de contraste): a mesma pessoa sob luz diferente fica parecida.
4. Normalização para [0, 1].

> **Nota metodológica — pistas falsas.** Uma CNN aprende qualquer diferença sistemática entre as classes, inclusive as que não têm nada a ver com o rosto. As versões anteriores tinham três:
> - **Domínio da foto**: com a base Olivetti (retratos de estúdio) a rede separava "retrato" de "webcam", não identidades — 100% dos rostos de fora viravam integrantes. Trocada pelo LFW.
> - **Borda verde**: o `captura_faces.py` antigo desenhava o retângulo do detector no quadro *antes* de recortar; 735 das 839 fotos de integrantes tinham a borda e nenhuma foto de desconhecido. Na demo (sem borda) o integrante tendia a virar "Desconhecido". Corrigido no script de captura atual; as fotos atuais já são recortadas sem a borda.
> - **Fundo e luz do ambiente**: todas as fotos de integrantes vêm da mesma webcam e dos mesmos cômodos, e nenhuma de desconhecido. "Este fundo / esta câmera" vira pista de integrante — a explicação mais provável para um amigo filmado ali às vezes sair com o nome de alguém do grupo. O recorte alinhado corta o fundo, o CLAHE reduz o efeito da luz e as pastas de `dados/desconhecidos/` capturadas na mesma webcam (`--desconhecido`) ensinam que o ambiente não é a pessoa.

In [ ]:
def carregar_fotos(pasta):
    # Toda imagem da pasta vira uma matriz 64x64 em tons de cinza, normalizada, pela
    # mesma função da demo. Cada foto recebe um "grupo": fotos do mesmo grupo ficam
    # sempre do mesmo lado da divisão treino/teste.
    arquivos = (
        sorted(pasta.glob("*.jpg"))
        + sorted(pasta.glob("*.jpeg"))
        + sorted(pasta.glob("*.png"))
    )
    fotos, grupos = [], []
    for posicao, arquivo in enumerate(arquivos):
        fotos.append(preparar_para_cnn(ler_imagem(arquivo), IMG_SIZE))
        if "__" in arquivo.stem:   # LFW (<pessoa>__<n>.jpg): o grupo é a pessoa
            grupos.append(f"{pasta.name}/{arquivo.stem.split('__')[0]}")
        else:                      # webcam/vídeo: fotos seguidas são quase iguais
            grupos.append(f"{pasta.name}/{posicao // TAMANHO_BLOCO}")
    if not fotos:
        raise ValueError(f"Nenhuma foto em {pasta} — rode src/captura_faces.py primeiro.")
    return np.stack(fotos), grupos


fotos_integrantes, grupos_integrantes = {}, {}
for nome in NOMES:
    fotos_integrantes[nome], grupos_integrantes[nome] = carregar_fotos(PASTA_INTEGRANTES / nome)
    print(f"{nome}: {len(fotos_integrantes[nome])} fotos")

In [ ]:
# Classe "Desconhecido": pessoas que NÃO são do grupo.
# LFW: rostos de gente real, com cabelo, óculos, iluminação e fundos variados. As fotos
# originais (250x250) passam pelo MESMO detector + alinhamento da webcam e ficam
# guardadas em dados/desconhecidos/lfw/ -- isso só roda na primeira execução.
if not PASTA_LFW.exists():
    fetch_lfw_people(min_faces_per_person=70)   # só baixa a base, se ainda não estiver no PC
    origem = Path(get_data_home()) / "lfw_home" / "lfw_funneled"
    pessoas = sorted(p for p in origem.iterdir() if p.is_dir())
    np.random.default_rng(SEED).shuffle(pessoas)

    detector = criar_detector()
    PASTA_LFW.mkdir(parents=True)
    salvas = 0
    for pessoa in pessoas:
        for arquivo in sorted(pessoa.glob("*.jpg"))[:LFW_POR_PESSOA]:
            img = ler_imagem(arquivo)
            rostos = detectar_rostos(detector, img)
            if not rostos:
                continue
            # a pessoa rotulada é a do centro da foto (às vezes há outras atrás)
            centro = np.array(img.shape[1::-1]) / 2
            rosto = min(rostos, key=lambda r: np.abs(r[:2] + r[2:4] / 2 - centro).sum())
            cv2.imwrite(str(PASTA_LFW / f"{pessoa.name}__{salvas:04d}.jpg"), alinhar(img, rosto))
            salvas += 1
        if salvas >= N_DESCONHECIDO_LFW:
            break

# todas as pastas de dados/desconhecidos/: o LFW e o WIDER cobrem a variedade
# de identidades; as capturadas com a webcam do grupo cobrem o ambiente da demo
PASTAS_DESCONHECIDO = sorted(p for p in PASTA_DESCONHECIDOS.iterdir() if p.is_dir())
fotos_desconhecido, grupos_desconhecido = [], []
for pasta in PASTAS_DESCONHECIDO:
    fotos, grupos_pasta = carregar_fotos(pasta)
    fotos_desconhecido.append(fotos)
    grupos_desconhecido += grupos_pasta
    print(f"Desconhecido / {pasta.name}: {len(fotos)} fotos")
fotos_desconhecido = np.concatenate(fotos_desconhecido)

# Monta X, y e os grupos: cada integrante vira uma classe; os de fora, a classe extra
X = np.concatenate([fotos_integrantes[nome] for nome in NOMES] + [fotos_desconhecido])
y = np.concatenate(
    [np.full(len(fotos_integrantes[nome]), rotulo) for rotulo, nome in enumerate(NOMES)]
    + [np.full(len(fotos_desconhecido), len(NOMES))]
).astype(np.int64)
grupos = np.array(sum((grupos_integrantes[nome] for nome in NOMES), []) + grupos_desconhecido)

print("Total de imagens:", len(X), "| Formato de cada imagem:", X.shape[1:])

In [ ]:
# Cadeia de pré-processamento de UMA foto: do arquivo salvo até a entrada da CNN
exemplo_arquivo = sorted((PASTA_INTEGRANTES / NOMES[0]).glob("*.jpg"))[0]
original = ler_imagem(exemplo_arquivo)
cinza_64 = cv2.resize(
    cv2.cvtColor(original, cv2.COLOR_BGR2GRAY), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA
)

fig, eixos = plt.subplots(1, 3, figsize=(9, 3))
eixos[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
eixos[0].set_title(f"1) recorte alinhado {original.shape[1]}x{original.shape[0]}", fontsize=9)
eixos[1].imshow(cinza_64, cmap="gray", vmin=0, vmax=255)
eixos[1].set_title(f"2) cinza {IMG_SIZE}x{IMG_SIZE}", fontsize=9)
eixos[2].imshow(preparar_para_cnn(original, IMG_SIZE), cmap="gray", vmin=0, vmax=1)
eixos[2].set_title("3) CLAHE, pixels em [0, 1]", fontsize=9)
for eixo in eixos:
    eixo.axis("off")
plt.suptitle("Imagens intermediárias do pré-processamento")
plt.tight_layout()
plt.show()

In [ ]:
# O que a CNN vê: 5 exemplos de cada classe
fig, eixos = plt.subplots(len(CLASSES), 5, figsize=(10, 2 * len(CLASSES)))
for linha, classe in enumerate(range(len(CLASSES))):
    for eixo, i in zip(eixos[linha], np.flatnonzero(y == classe)[:5]):
        eixo.imshow(X[i], cmap="gray", vmin=0, vmax=1)
        eixo.axis("off")
    eixos[linha, 0].set_title(CLASSES[classe], fontsize=10, loc="left")

plt.suptitle("Entradas da CNN: 64x64 tons de cinza, 5 exemplos por classe")
plt.tight_layout()
plt.show()

## 2. Divisão treino/teste e balanceamento

Divisão **estratificada** (mesma proporção de cada classe no treino e no teste) e **por grupos** (`StratifiedGroupKFold`):

- A captura automática tira ~3 fotos por segundo, então fotos vizinhas são quase idênticas. Numa divisão foto a foto, a "gêmea" de cada foto de teste fica no treino e a acurácia de teste mede memória, não generalização (a versão anterior marcava 99,4% e mesmo assim errava ao vivo). Aqui cada bloco de 20 fotos seguidas vai inteiro para um lado só.
- No LFW o grupo é a **pessoa**: os desconhecidos do teste são pessoas que a rede **nunca viu** — o mesmo caso do amigo na frente da webcam.

In [ ]:
# 10 dobras estratificadas por grupo; 3 delas (30%) formam o teste
dobras = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED).split(X, y, grupos))
teste = np.concatenate([indices for _, indices in dobras[:round(TEST_SIZE * 10)]])
treino = np.setdiff1d(np.arange(len(X)), teste)
assert not set(grupos[treino]) & set(grupos[teste])

X_train, X_test, y_train, y_test = X[treino], X[teste], y[treino], y[teste]
grupos_train = grupos[treino]


def contagem_por_classe(rotulos):
    return pd.Series(rotulos).value_counts().reindex(range(len(CLASSES)), fill_value=0)


contagens = pd.DataFrame({
    "Completo": contagem_por_classe(y),
    "Treino (70%)": contagem_por_classe(y_train),
    "Teste (30%)": contagem_por_classe(y_test),
})
contagens.index = CLASSES
percentuais = (contagens / contagens.sum() * 100).round(2)

print("Contagens por classe:")
display(contagens)
print("Percentuais por classe (%):")
display(percentuais)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

contagem_por_classe(y_train).plot.bar(ax=eixos[0], color="steelblue", edgecolor="black")
eixos[0].set_title("Treino (70%)")

contagem_por_classe(y_test).plot.bar(ax=eixos[1], color="darkorange", edgecolor="black")
eixos[1].set_title("Teste (30%)")

for eixo in eixos:
    eixo.set_xticks(range(len(CLASSES)))
    eixo.set_xticklabels(CLASSES, rotation=0)
    eixo.set_ylabel("Quantidade de imagens")

plt.suptitle("Distribuição das classes (integrantes + Desconhecido)")
plt.tight_layout()
plt.show()

## 3. Arquitetura da CNN

Rede usada (entrada 64x64, 1 canal):

1. **Aumento de dados** (`RandomFlip`, `RandomRotation`, `RandomTranslation`, `RandomZoom`, `RandomBrightness`, `RandomContrast`): só atuam durante o treino; criam variações artificiais de posição e de **iluminação** e reduzem o overfitting.
2. **Conv2D 32 filtros 3x3 + ReLU**: detecta bordas e contrastes simples.
3. **MaxPooling 2x2**: reduz a resolução pela metade, mantendo o que importa.
4. **Conv2D 64 + MaxPooling**: combina bordas em partes do rosto (olhos, nariz, boca).
5. **Conv2D 128 + MaxPooling** (duas vezes): padrões maiores, mais abstratos (combinação de partes). A 4ª convolução deixa o mapa final em 4x4 e corta pela metade os pesos da camada densa.
6. **Flatten + Dense 128 + ReLU**: transforma os mapas em uma camada de decisão.
7. **Dropout 0.5**: desliga metade dos neurônios aleatoriamente no treino (regularização).
8. **Dense 4 + Softmax**: probabilidade final para cada uma das 4 classes.

In [ ]:
aumento = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),                   # espelha horizontalmente
        layers.RandomRotation(0.03),                       # giros leves (até ~11 graus)
        layers.RandomTranslation(0.05, 0.05),              # pequenos erros do alinhamento
        layers.RandomZoom(0.10),                           # zoom leve
        layers.RandomBrightness(0.2, value_range=(0, 1)),  # mais claro / mais escuro
        layers.RandomContrast(0.3, value_range=(0, 1)),    # mais / menos contraste
    ],
    name="aumento",
)

modelo = keras.Sequential(
    [
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        aumento,
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(len(CLASSES), activation="softmax"),
    ],
    name=NOME_MODELO,   # a demo confere o nome: modelo de notebook antigo é recusado
)

modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
modelo.summary()

In [ ]:
# Efeito do aumento de dados: original (linha de cima) x versões sorteadas (linha de baixo)
amostra = X_train[:6]
aumentadas = aumento(amostra[..., None], training=True).numpy()[..., 0]

fig, eixos = plt.subplots(2, 6, figsize=(12, 4))
for coluna in range(6):
    eixos[0, coluna].imshow(amostra[coluna], cmap="gray", vmin=0, vmax=1)
    eixos[1, coluna].imshow(aumentadas[coluna], cmap="gray", vmin=0, vmax=1)
for eixo in eixos.ravel():
    eixo.axis("off")
eixos[0, 0].set_title("original", loc="left", fontsize=10)
eixos[1, 0].set_title("após o aumento", loc="left", fontsize=10)

plt.suptitle("Aumento de dados (aplicado só durante o treino)")
plt.tight_layout()
plt.show()

## 4. Treino

- Separação de **validação** estratificada e por grupos (20% do treino) só para o `EarlyStopping` decidir quando parar — **o conjunto de teste fica intocado até a avaliação final**.
- `EarlyStopping(patience=8)`: para quando a perda de validação para de melhorar e devolve os melhores pesos.

In [ ]:
# validação também por grupos: com fotos "gêmeas" dos dois lados, a perda de
# validação ia a ~0 e o EarlyStopping não tinha o que medir
indices_tr, indices_val = next(
    StratifiedGroupKFold(n_splits=round(1 / VAL_SIZE), shuffle=True, random_state=SEED)
    .split(X_train, y_train, grupos_train)
)
X_tr, X_val = X_train[indices_tr], X_train[indices_val]
y_tr, y_val = y_train[indices_tr], y_train[indices_val]

parada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)

historico = modelo.fit(
    X_tr[..., None],
    y_tr,
    validation_data=(X_val[..., None], y_val),
    epochs=EPOCAS,
    batch_size=BATCH_SIZE,
    callbacks=[parada],
    verbose=1,
)

In [ ]:
epocas_rodadas = range(1, len(historico.history["loss"]) + 1)

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))

eixos[0].plot(epocas_rodadas, historico.history["loss"], label="treino")
eixos[0].plot(epocas_rodadas, historico.history["val_loss"], label="validação")
eixos[0].set_title("Loss (erro)")
eixos[0].set_xlabel("Época")
eixos[0].legend()

eixos[1].plot(epocas_rodadas, historico.history["accuracy"], label="treino")
eixos[1].plot(epocas_rodadas, historico.history["val_accuracy"], label="validação")
eixos[1].set_title("Acurácia")
eixos[1].set_xlabel("Época")
eixos[1].legend()

plt.suptitle("Curvas de treino da CNN")
plt.tight_layout()
plt.show()

## 5. Avaliação no conjunto de teste

In [ ]:
probabilidades = modelo.predict(X_test[..., None], verbose=0)
y_pred = np.argmax(probabilidades, axis=1)

metricas = pd.DataFrame([
    {
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }
], index=["CNN"])

display(metricas.style.format("{:.2%}"))

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASSES,
    yticklabels=CLASSES,
)
plt.xlabel("Previsto")
plt.ylabel("Verdadeiro")
plt.title("CNN — Matriz de confusão (teste)")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3, zero_division=0))

### Limiar de confiança da demo

A demo troca o nome por "Desconhecido" quando a maior probabilidade fica abaixo de `CONFIANCA_MINIMA`. A tabela mostra o efeito de cada limiar no conjunto de teste:

- **Integrante reconhecido**: fração das fotos de integrantes que sai com o nome certo (quanto maior, melhor).
- **Desconhecido aceito como integrante**: fração das fotos de gente de fora que sai com o nome de alguém do grupo — o erro da pessoa de fora virar um integrante (quanto menor, melhor).

Subir o limiar troca um erro pelo outro. O teste não tem a webcam nem o ambiente da demo, então a tabela é o ponto de partida; o ajuste fino é ao vivo.

In [ ]:
eh_integrante = y_test < len(NOMES)
confianca = probabilidades.max(axis=1)

linhas = []
for limiar in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    aceito = (y_pred < len(NOMES)) & (confianca >= limiar)
    linhas.append({
        "Limiar": limiar,
        "Integrante reconhecido": (aceito & (y_pred == y_test))[eh_integrante].mean(),
        "Desconhecido aceito como integrante": aceito[~eh_integrante].mean(),
    })

display(pd.DataFrame(linhas).set_index("Limiar").style.format("{:.1%}"))

In [ ]:
rng = np.random.default_rng(SEED)
escolhidos = rng.choice(len(X_test), size=12, replace=False)

fig, eixos = plt.subplots(3, 4, figsize=(10, 8))
for eixo, i in zip(eixos.ravel(), escolhidos):
    eixo.imshow(X_test[i], cmap="gray", vmin=0, vmax=1)
    acertou = y_pred[i] == y_test[i]
    eixo.set_title(
        f"{CLASSES[y_pred[i]]}\n(real: {CLASSES[y_test[i]]})",
        color="green" if acertou else "red",
        fontsize=9,
    )
    eixo.axis("off")

plt.suptitle("Previsto no título — verde = acerto, vermelho = erro")
plt.tight_layout()
plt.show()

## 6. Imagens intermediárias da CNN

As **ativações** mostram o que cada convolução "enxerga". A primeira camada realça bordas e contrastes; as seguintes combinam isso em partes do rosto. É a mesma imagem de entrada sendo transformada camada a camada.

In [ ]:
# Modelo auxiliar que devolve a saída de cada convolução para uma mesma imagem
extrator = keras.Model(
    modelo.inputs,
    [camada.output for camada in modelo.layers if isinstance(camada, layers.Conv2D)],
)

i = int(np.flatnonzero(y_pred == y_test)[0])
mapas = extrator(X_test[i][None, ..., None])
print(f"Exemplo: real = {CLASSES[y_test[i]]} | previsto = {CLASSES[y_pred[i]]} "
      f"({probabilidades[i].max():.0%} de confiança)")

fig, eixos = plt.subplots(len(mapas), 8, figsize=(12, 1.7 * len(mapas)))
for linha, mapa in enumerate(mapas):
    for coluna in range(8):
        eixos[linha, coluna].imshow(mapa[0, ..., coluna], cmap="viridis")
        eixos[linha, coluna].axis("off")
    eixos[linha, 0].set_title(f"Conv {linha + 1} — {mapa.shape[-1]} canais", loc="left", fontsize=9)

plt.suptitle(f"Ativações das convoluções para um rosto de '{CLASSES[y_test[i]]}' (8 canais por camada)")
plt.tight_layout()
plt.show()

## 7. Discussão

**O que esperar**

- Com a divisão por grupos, a acurácia de teste deixa de medir memória: são fotos de momentos que a rede não viu e desconhecidos que ela nunca viu. O desafio real continua sendo a **generalização ao vivo** (luz, posição e distância diferentes das fotos de coleta).
- **Limitação**: os desconhecidos do LFW e do WIDER vêm de outras câmeras. Enquanto só os integrantes forem fotografados com a webcam do grupo, "jeito desta webcam / deste cômodo" continua sendo pista de integrante. A correção é ter pastas em `dados/desconhecidos/` capturadas (`--desconhecido`) com a mesma webcam e nos mesmos ambientes da coleta e da demo.
- Se as curvas de treino e validação se afastarem muito (overfitting), os caminhos são: coletar fotos mais **variadas** (outras sessões de luz, outras poses), aumentar a intensidade do aumento de dados ou aumentar o `Dropout`.

**Na demo ao vivo** (`src/demo_reconhecimento.py`) há um segundo filtro: `CONFIANCA_MINIMA` (ver a tabela de limiares acima).

- Mesmo se a CNN apontar um nome, quando a probabilidade for menor que esse limiar a resposta vira "Desconhecido".
- A demo faz a média das últimas 10 leituras e zera essa média quando nenhum rosto aparece — sem isso, quem entrava logo depois herdava a média da pessoa anterior.

## 8. Conclusão

- Uma CNN pequena (4 convoluções, ~500 mil parâmetros) e uma base modesta (~200 fotos por pessoa) reconhecem os integrantes e marcam o resto como **"Desconhecido"** — desde que treino e demo vejam o mesmo tipo de imagem (mesmo detector, alinhamento e pré-processamento) e nenhuma pista fora do rosto separe as classes.
- Diferente da atividade anterior (KNN/K-Means/DBSCAN, que iam direto nos pixels), aqui a **CNN aprende sozinha** os filtros relevantes (bordas → partes do rosto → padrões), o que a torna adequada para imagens.
- O sistema tem duas camadas de proteção contra falsos positivos: a **4ª classe treinada** e o **limiar de confiança** da demo.
- Próximos passos naturais: mais "desconhecidos" reais capturados com a mesma webcam (colegas de turma) e coleta em mais sessões de luz.

In [ ]:
ARQUIVO_MODELO.parent.mkdir(parents=True, exist_ok=True)
modelo.save(ARQUIVO_MODELO)

ARQUIVO_CLASSES.write_text("\n".join(CLASSES), encoding="utf-8")

print(f"Salvos em modelos/: {ARQUIVO_MODELO.name} e {ARQUIVO_CLASSES.name} — a demo usa os dois.")